# Lab 08: 長短期記憶網路 (LSTM) 實作 — 真實電影評論情感分析

在本單元中，我們將解決傳統 RNN 的「健忘」問題。我們將實作一個 **LSTM (Long Short-Term Memory) 神經網路**。
我們將從公開網路上下載 **IMDb 真實電影評論資料集 (IMDb Movie Reviews Dataset)**，學習自然語言處理的標準前處理流程：字詞切分 (Tokenization)、建立字表 (Vocabulary)、長度對齊 (Padding)、詞嵌入 (Word Embedding)，並完成情感分類器的訓練與測試！

### 任務 1: 從公開網路載入真實電影評論資料集
我們使用 `pandas` 直接從公開 GitHub 載入 IMDb 評論的 CSV 檔案。為了確保在課堂上訓練快速，我們只取前 **1,000 筆** 資料作為訓練集。
*(貼心設計：若電腦教室因為網路限制下載失敗，會自動切換為本地預設資料集，確保教學不受影響！)*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import urllib.request

# 預設本地備用極簡資料集 (18筆)
local_backup_data = [
    ("this movie was great and wonderful", 1),
    ("i loved the acting and the plot", 1),
    ("best movie of the year fantastic", 1),
    ("an absolute masterpiece highly recommend", 1),
    ("beautiful story great acting loved it", 1),
    ("excellent film with brilliant cast", 1),
    ("amazing visual effects and great music", 1),
    ("very clean comedy highly entertaining", 1),
    ("perfect ending emotional and satisfying", 1),
    ("this movie was terrible and waste of time", 0),
    ("worst acting and boring story", 0),
    ("i hated this film it was awful", 0),
    ("very disappointing and bad script", 0),
    ("dumb plot waste of money not recommended", 0),
    ("worst movie ever very boring acting", 0),
    ("highly predictable and extremely stupid", 0),
    ("failed to impress very poor execution", 0),
    ("it was a painful waste of my night", 0)
]

imdb_url = "https://raw.githubusercontent.com/LawrenceDuan/IMDb-Review-Analysis/master/IMDb_Reviews.csv"

try:
    print("正在從公開網路下載 IMDb 電影評論資料集 (約 20MB)...")
    # 使用 pandas 讀取公開 CSV 檔案
    df = pd.read_csv(imdb_url)
    print("下載完成！")
    # 選取前 1000 筆資料來訓練，避免課堂上跑太久
    df_subset = df.head(1000)
    # 取出評論 text 與標籤 sentiment (0=negative, 1=positive)
    raw_data = list(zip(df_subset['review'], df_subset['sentiment']))
    print(f"已成功載入前 {len(raw_data)} 筆真實電影評論。")
except Exception as e:
    print(f"網路下載失敗 ({e})。啟動本地備用極簡資料集...")
    raw_data = local_backup_data

# 顯示第一筆資料範例
print("\n--- 第一筆評論資料範例 ---")
print("文字評論:", raw_data[0][0][:150] + "...")
print("情感標籤:", "好評" if raw_data[0][1] == 1 else "差評")

### 任務 2: 文字前處理與字表建立
電腦看不懂英文單字，所以我們要：
1. 將句子切開成一個個單字 (Tokenization)。
2. 收集所有不重複單字建立對照表 (Vocabulary)，並加入未知字 `<UNK>` 與補齊長度字 `<PAD>`。
3. 將每句話轉換為數字序列，並將長度「補齊/裁剪」至相同長度 (長度對齊 (Padding))。

In [ ]:
# 建立字詞切分器 (簡單按空白鍵切分單字，並移除標點符號)
def tokenize(text):
    clean_text = "".join([c.lower() for c in text if c.isalnum() or c.isspace()])
    return clean_text.split()

# 建立字表
vocab = {"<PAD>": 0, "<UNK>": 1} # PAD 用來對齊長度，UNK 用來代表未見過的生字
for text, _ in raw_data:
    for word in tokenize(text):
        if word not in vocab:
            vocab[word] = len(vocab)

vocab_size = len(vocab)
print(f"不重複字表大小 (Vocab Size): {vocab_size}")

# 設定每句話的固定長度為 30 個單字 (真實評論較長，所以設定長一點)
max_length = 30

def text_to_sequence(text, vocab, max_len):
    tokens = tokenize(text)
    # 轉為數字 ID，如果不在字表就用 UNK (1)
    seq = [vocab.get(token, 1) for token in tokens]

    # 長度對齊 (Padding)：把每句話對齊成相同長度 max_len
    if len(seq) > max_len:
        # 太長 → 截斷成前 max_len 個
        seq = seq[:max_len]
    else:
        # 太短 → 在後面補 0 (代表 <PAD>)，直到長度為 max_len
        seq = seq + [0] * (max_len - len(seq))
    return seq

# 轉換所有資料
X_data = [text_to_sequence(text, vocab, max_length) for text, _ in raw_data]
y_data = [label for _, label in raw_data]

# 轉為 PyTorch 張量
X_tensor = torch.tensor(X_data, dtype=torch.long)
y_tensor = torch.tensor(y_data, dtype=torch.float32).unsqueeze(1)

print("第一筆轉換與 長度對齊 (Padding) 後張量:", X_tensor[0])

### 任務 3: 定義 LSTM 情感分類模型
LSTM 模型包含：
1. **字詞空間化 (Embedding) 層**：把單字 ID 轉換成帶有語意方向的稠密向量（例如 $32$ 維）。
2. **LSTM 層**：處理序列向量，輸出隱藏狀態。
3. **線性層與 Sigmoid**：將最後一個時間步的隱藏狀態做分類，輸出好評機率 (0~1)。

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SentimentLSTM, self).__init__()
        # 1. 詞嵌入層
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # 2. TODO: 定義一個 nn.LSTM 層，輸入維度為 embedding_dim，隱藏維度為 hidden_dim，batch_first=True
        # 提示：self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        # ??? (請在此填寫你的答案)
        
        # 3. 輸出層與 Sigmoid 分類器
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # x 形狀: [Batch, seq_len]
        embedded = self.embedding(x) # 轉為 [Batch, seq_len, embedding_dim]
        
        # 通過 LSTM，取得輸出與最後的 (hidden, cell) 狀態
        lstm_out, (hn, cn) = self.lstm(embedded)
        
        # (已幫你完成) 提取序列最後一個時間步的輸出 (Last time step)
        # 形狀為 [Batch, hidden_dim]
        last_out = lstm_out[:, -1, :]
        
        out = self.sigmoid(self.fc(last_out))
        return out

emb_dim = 32
hidden_dim = 32
model = SentimentLSTM(vocab_size, emb_dim, hidden_dim)
print(model)

### 任務 4: 訓練模型
我們使用 Adam 智慧調整器 (Optimizer)在設備（GPU / CPU）上訓練模型。

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
X_tensor, y_tensor = X_tensor.to(device), y_tensor.to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

print(f"開始在 {device} 上訓練 LSTM 情感分類模型...")
for epoch in range(150):
    predictions = model(X_tensor)
    loss = criterion(predictions, y_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 25 == 0:
        # 計算準確率
        binary_preds = (predictions >= 0.5).float()
        acc = (binary_preds == y_tensor).float().mean().item() * 100
        print(f"Epoch [{epoch+1}/150], Loss: {loss.item():.4f}, Accuracy: {acc:.1f}%")

### 任務 5: 學生自己寫影評，測試模型！
現在，學生可以自行在下方輸入任何英文短句，測試訓練好的 LSTM 能否看懂它的情緒！

In [ ]:
def predict_sentiment(test_sentence):
    model.eval()
    with torch.no_grad():
        # 前處理輸入的句子
        seq = text_to_sequence(test_sentence, vocab, max_length)
        input_tensor = torch.tensor([seq], dtype=torch.long).to(device)
        
        # 預測
        prob = model(input_tensor).item()
        sentiment = "好評 (Positive)" if prob >= 0.5 else "差評 (Negative)"
        print(f"評論: 『{test_sentence}』")
        print(f"好評機率: {prob*100:.1f}% → 預測情感: {sentiment}\n")

# (可自由修改) 在下方改寫測試句子，測測看模型準不準！
predict_sentiment("terrible plot and extremely boring")
predict_sentiment("i hated the characters it was a complete waste of time")

---

## Lab 08 學習重點小結
### 核心概念掌握
- LSTM 透過遺忘門、輸入門、輸出門三大閥門控制記憶的保留與更新。
- 詞嵌入 (Word Embedding) 將詞彙映射到連續向量空間，語義相近的詞距離近。
- 情感分析是 NLP 的基礎任務：將文字序列分類為正面/負面情緒。
- 長度對齊 (Padding) + Packing：對不等長序列補齊後，再使用 pack_padded_sequence 加速 LSTM。

### 快速自評測驗

**Q1. LSTM 比標準 RNN 更能學習長期依賴的原因是？**
- A. LSTM 參數量更大
- B. LSTM 的閥門機制可選擇性保留或遺忘資訊
- C. LSTM 使用了更多的活化函數

<details><summary>查看解答</summary>

**答案：B — 遺忘門、輸入門讓 LSTM 能動態控制記憶，緩解梯度消失。**

</details>

**Q2. 詞嵌入 (Word Embedding) 的主要作用為？**
- A. 將詞彙轉為高維稀疏向量 (One-Hot)
- B. 將詞彙轉為低維稠密向量，捕捉語義關係
- C. 對詞彙進行分詞

<details><summary>查看解答</summary>

**答案：B — 字詞空間化 (Embedding) 讓相似詞彙在向量空間中距離更近，如「國王」與「女王」。**

</details>

### 完成確認清單
在繼續下一個 Lab 前，請確認你能做到：
- [ ] 我可以用一句話解釋「LSTM 透過遺忘門、輸入門、輸出門三大閥門控制記憶的保留與」
- [ ] 我可以用一句話解釋「詞嵌入 (Word Embedding) 將詞彙映射到連續向」
- [ ] 我可以用一句話解釋「情感分析是 NLP 的基礎任務」
- [ ] 我的程式碼成功執行並得到預期輸出